# 01 — Lagged cross-correlation screen

**docs/04 §4.1.** Kelp anomaly at quarter *t* against every environmental feature
anomaly at *t−0…4*, per polygon × environmental series. The first rung of the
docs/04 §4 analysis ladder, and the one that decides which relationships are
worth carrying to §4.3.

**This is a screen, and screening claims nothing.** docs/04 §5 makes §4.1
exploratory: it ranks candidates, it does not test them. So this notebook
reports correlation coefficients, sample sizes and an autocorrelation-adjusted
effective sample size — and **no p-values**, deliberately. The lag × feature ×
polygon grid generates hundreds of coefficients; a p-value column would invite
exactly the reading docs/04 §5 forbids.

**Reproducibility.** Runs top to bottom from `features/comparison.parquet` and
nothing else — no side reads of `observations/` or `raw/`. Nothing here is
stochastic, so there is no seed to set. Every table is stamped with the SHA-256
of the comparison file it was computed from; quote that digest in any figure
caption or write-up.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

from kelpcompare.features.comparison import COMPARISON_KEY
from kelpcompare.features.config import load_feature_config
from kelpcompare.features.kelp import MEASURED

pd.set_option("display.width", 200)


def repo_root() -> Path:
    """The checkout root, so the notebook runs from anywhere."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "features" / "comparison.parquet").exists():
            return candidate
    raise FileNotFoundError(
        "no data/features/comparison.parquet above the working directory -- "
        "run `kelpcompare features` first"
    )


COMPARISON = repo_root() / "data" / "features" / "comparison.parquet"
DIGEST = hashlib.sha256(COMPARISON.read_bytes()).hexdigest()

comparison = pd.read_parquet(COMPARISON)

# Which columns identify one series, and which anomalies belong to the kelp half,
# are facts about the schema (docs/03) -- so they are read from the package that
# wrote the table rather than restated here. A restated key is how an analysis
# silently pools two depths into one series, or screens a kelp feature as an
# environmental one, the first time the schema widens. It has widened once
# already: docs/03 records an earlier draft keyed without parameter or depth.
SERIES_KEY = [column for column in COMPARISON_KEY if column not in {"year", "quarter"}]
ENV_SERIES_KEY = [
    column for column in SERIES_KEY if column not in {"polygon_id", "kelp_source", "lag"}
]
KELP_ANOMALIES = {f"{name}_anom" for name in MEASURED}

absent = [
    column
    for column in (*SERIES_KEY, "year", "quarter", *sorted(KELP_ANOMALIES))
    if column not in comparison.columns
]
if absent:
    raise ValueError(
        f"the installed kelpcompare does not describe this table -- {absent} absent. "
        "Rebuild it with `kelpcompare features`, or check out the package this table "
        "was written by."
    )

# Which parameters may be pre-registered is a docs/04 s5 decision, so it is read
# from `features.json` -- the registry that records it, with its reasons -- rather
# than restated here as a literal. Restating it is how a demotion agreed in the
# registry stays silently absent from the analysis that is supposed to honour it.
ROLES = load_feature_config(repo_root() / "data" / "registry" / "features.json").roles()

print(f"comparison.parquet   sha256:{DIGEST[:16]}   {len(comparison):,} rows")
print(f"Kelp Watch revision  {sorted(set(comparison['kelp_watch_revision']))}")
print(f"polygons             {comparison['polygon_id'].nunique()}")
print(f"lags                 {sorted(set(comparison['lag']))}")
print(f"series key           {' x '.join(SERIES_KEY)}")
print(f"predictors           {', '.join(n for n, r in ROLES.items() if r == 'predictor')}")
print(f"controls             {', '.join(n for n, r in ROLES.items() if r == 'control')}")

comparison.parquet   sha256:4cde6f9d95207dc1   51,000 rows
Kelp Watch revision  [23]
polygons             6
lags                 [0, 1, 2, 3, 4]
series key           polygon_id x kelp_source x env_source x site_id x parameter x depth_m x lag
predictors           sea_water_temperature, water_level, wave_significant_height, wave_peak_period
controls             air_temperature, wind_speed


## 1. The gate

`comparison.parquet` keeps every row, including those where either side is
unusable or where a lag reaches past the start of the environmental record —
docs/03 makes `usable` the single gate so that what filtering costs is visible
rather than already spent.

Spending it here, once, and reporting what it cost.

In [2]:
# Which kelp anomaly answers the question is an analysis choice, so it stays a
# literal here -- docs/04 s2: a bed can thin without shrinking and shrink without
# thinning, and the notebook picks. Which columns are *kelp* is not a choice.
KELP = "kelp_area_m2_anom"
ENV_FEATURES = [
    column
    for column in comparison.columns
    if column.endswith("_anom") and column not in KELP_ANOMALIES
]

usable = (
    comparison["kelp_usable"] & comparison["env_usable"].fillna(False) & comparison[KELP].notna()
)
gated = comparison[usable]

attrition = pd.Series(
    {
        "all rows": len(comparison),
        "kelp side usable": int(comparison["kelp_usable"].sum()),
        "...and an environmental row exists": int(
            (comparison["kelp_usable"] & comparison["env_usable"].notna()).sum()
        ),
        "...and it is usable too": int(
            (comparison["kelp_usable"] & comparison["env_usable"].fillna(False)).sum()
        ),
        "...and the kelp anomaly is not null": len(gated),
    },
    name="rows",
)
print(attrition.to_string())
print()
missing_env = comparison["kelp_usable"] & comparison["env_usable"].isna()
env_start = int(comparison.loc[comparison["env_usable"].notna(), "env_year"].min())
unreachable = int((missing_env & (comparison["year"] < env_start)).sum())

print()
print(f"{int(missing_env.sum()):,} kelp-usable rows have no environmental row at all, and")
print(f"{unreachable:,} of those are kelp quarters before {env_start}, which no lag could have")
print(f"reached -- the kelp record starts in {int(comparison['year'].min())}, the environmental")
print(
    f"one in {env_start}. Only {int(missing_env.sum()) - unreachable:,} are the lag itself reaching"
)
print("back past the start of the environmental record.")

all rows                               51000
kelp side usable                       48085
...and an environmental row exists     20983
...and it is usable too                20074
...and the kelp anomaly is not null    20074


27,102 kelp-usable rows have no environmental row at all, and
0 of those are kelp quarters before 1983, which no lag could have
reached -- the kelp record starts in 1984, the environmental
one in 1983. Only 27,102 are the lag itself reaching
back past the start of the environmental record.


## 2. Which features are worth screening at all

Two different reasons a feature should not enter the screen, and they must not be
confused with each other.

**Not applicable.** `quarterly_env` is wide and sparse by design (docs/03): a row
carries whichever features its `feature_set` defines and null elsewhere. Only
`sea_water_temperature` gets the docs/04 §2 ecological set, so
`air_temperature × days_above_20c` is not a weak feature — it is not a feature.

**Degenerate.** A feature whose values are pinned to a sensor's resolution floor
produces anomalies that encode nothing, and correlating them generates
coefficients that look strong and mean nothing.

That second one is not hypothetical here. **Quarterly minimum wind speed at
LJAC1 takes two values across the whole record** — `0.0` in 51 quarters and
`0.1` in 17 — because the anemometer cannot resolve below that. Its anomaly
takes four, being those two values against differing quarterly baselines, which
is the number the audit below counts; either way it is a disguised binary flag,
and in an earlier pass of this screen it
produced the largest coefficients in the entire grid, |r| up to 0.74, on an
effective sample size below 10.

The rule applied below, stated so it can be argued with:

- **not applicable** — no non-null values at all;
- **excluded** — fewer than 10 distinct anomaly values;
- **flagged `low_resolution`** — fewer than 25, kept but not to be read alone.

Distinct-value count rather than variance, because a genuine count feature like
`days_above_23c` has a legitimately small range and must not be thrown out for
it.

In [3]:
EXCLUDE_BELOW = 10
FLAG_BELOW = 25


def series_of(values) -> tuple:
    """A hashable series key that survives the null `depth_m` every met parameter has.

    `nan != nan`, so a tuple carrying a raw null does not match itself across two
    frames and a set lookup on it silently misses. `comparison.py` avoids the same
    trap by joining on a text key rather than on the columns; this is that, small.
    """
    return tuple("" if pd.isna(value) else str(value) for value in values)


# Keyed on the environmental *series*, not on `parameter` alone: a parameter is
# not a series, and two stations measuring it -- or one station at two depths --
# would otherwise be pooled into a single distinct-value count, so one degenerate
# anemometer could carry a good one out of the screen with it, or be hidden by it.
# `dropna=False` because `depth_m` is null for every met parameter, and pandas
# drops null group keys by default: without it this would silently discard
# `air_temperature` and `wind_speed` entirely.
audit = []
for key, block in gated.groupby(ENV_SERIES_KEY, dropna=False, sort=True):
    for column in ENV_FEATURES:
        values = block[column].dropna()
        audit.append(
            {
                **dict(zip(ENV_SERIES_KEY, key, strict=True)),
                "feature": column.removesuffix("_anom"),
                "present": len(values),
                "distinct": values.round(6).nunique(),
            }
        )
resolution = pd.DataFrame(audit)


def verdict(row) -> str:
    if row["present"] == 0:
        return "not applicable"
    if row["distinct"] < EXCLUDE_BELOW:
        return "excluded"
    return "low_resolution" if row["distinct"] < FLAG_BELOW else "ok"


resolution["verdict"] = resolution.apply(verdict, axis=1)

print(resolution["verdict"].value_counts().to_string())
print()
print("Not applicable -- the feature set does not define these for this parameter:")
na = resolution[resolution["verdict"] == "not applicable"]
print("   ", ", ".join(sorted(set(na["parameter"]))), "x the temperature feature set")
print()
print("Degenerate or low resolution, by the series they were measured on:")
print(
    resolution[resolution["verdict"].isin(["excluded", "low_resolution"])]
    .sort_values("distinct")
    .to_string(index=False)
)

dropped = resolution.loc[
    resolution["verdict"].isin(["not applicable", "excluded"]), [*ENV_SERIES_KEY, "feature"]
]
skip = {(*series_of(row[:-1]), row[-1]) for row in dropped.itertuples(index=False, name=None)}

verdict
ok                111
not applicable     53
low_resolution     19
excluded            4

Not applicable -- the feature set does not define these for this parameter:
    air_temperature, sea_water_temperature, wave_peak_period, wave_significant_height, wind_speed x the temperature feature set

Degenerate or low resolution, by the series they were measured on:
        env_source            site_id               parameter  depth_m        feature  present  distinct        verdict
              ndbc         NDBC:LJAC1              wind_speed      NaN            min     1482         4       excluded
              ndbc         NDBC:46254   sea_water_temperature     0.46 days_below_14c      162         7       excluded
              ndbc         NDBC:46254        wave_peak_period      NaN            max      162         7       excluded
              ndbc         NDBC:46254        wave_peak_period      NaN            p95      162         8       excluded
              ndbc         NDBC

## 3. The screen

One row per polygon × environmental series × feature × lag.

**`n_eff` is the column to read, not `n`.** Quarterly anomaly series are
autocorrelated, so the number of paired quarters overstates how much independent
evidence a coefficient rests on. `n_eff` applies the Quenouille/Bartlett
adjustment for the correlation between two autocorrelated series —

$$n_{\text{eff}} = n\,\frac{1 - r_1^{(x)} r_1^{(y)}}{1 + r_1^{(x)} r_1^{(y)}}$$

— which is a rough correction, not an exact one. It is here in place of p-values
precisely because it degrades gracefully: where two series are strongly
autocorrelated it collapses, and a coefficient sitting on `n_eff` of 8 announces
itself as worthless without anyone having to interpret a significance threshold.

**The lag-1 term is measured across calendar-adjacent quarters only.** The rows
this screen keeps are not the quarters the calendar has: a cloud-gapped kelp
quarter and an unusable environmental one each leave a hole, and
`air_temperature` and `wind_speed` have no Q2 anomaly at all, so their series
steps Q3, Q4, Q1, Q3. Treating each of those steps as lag 1 would measure
persistence across a hole, which biases the autocorrelation down and `n_eff`
correspondingly up — the one direction a ceiling must not be wrong in. A gap
breaks the pair rather than being bridged across it, on the same reasoning that
makes an unobserved day break a threshold spell (docs/04 §2) and makes the
QARTOD neighbour tests withdraw at a gap rather than guess (docs/04 §1). The
cell below reports what that costs.

**Read it as a ceiling on independent evidence, not an estimate of it.** Two
things bound it from the optimistic side. It is *capped at `n`*, because the
expression is symmetric in the sign of the lag-1 product and an anti-correlated
pair would otherwise be credited with more independent quarters than it has
quarters — uncapped it runs to infinity at a product of −1. And it *sees lag 1
only*: the kelp anomaly stays autocorrelated well past one quarter, so a
higher-order correction discounts the lag-4 cells further than this one does.
§6 measures both — how far past one quarter, and how much further — for the
cells the candidate rule selects, rather than quoting a figure worked out by
hand beside a table that cannot check it.

**`discounted` says whether the correction applied at all.** Where the lag-1
product comes out at or below zero there is nothing to discount, and `n_eff` is
the quarter count unchanged — which is right under the ceiling reading above, but
indistinguishable from a measured figure in the number alone. A cell at `n_eff`
50.0 that was never discounted and one at 50.6 that was are not the same kind of
50, and the column says which is which.

Both Pearson and Spearman are reported. A large gap between them is a sign the
association is driven by a handful of points, which at this sample size is the
common failure.

In [4]:
MIN_PAIRS = 12
MIN_ADJACENT = 3


def quarter_index(years: pd.Series, quarters: pd.Series) -> pd.Series:
    """A running quarter number, so "the quarter before" is arithmetic, not position."""
    return years * 4 + (quarters - 1)


def lag1(values: pd.Series, quarters: pd.Series) -> float:
    """Lag-1 autocorrelation, over calendar-adjacent quarters only.

    The rows reaching here are not the quarters the calendar has. A cloud-gapped
    kelp quarter and an unusable environmental one each leave a hole, and
    `air_temperature` and `wind_speed` have no Q2 anomaly at all, so their series
    steps Q3, Q4, Q1, Q3. A positional shift would call each of those steps lag 1
    and measure persistence across a hole -- which biases the autocorrelation
    down, and so `n_eff` up, in the one direction a ceiling must not be wrong in.

    So a gap breaks the pair rather than being bridged across, exactly as an
    unobserved day breaks a threshold spell (docs/04 s2), and where too few
    adjacent pairs survive this withdraws rather than guessing, as the QARTOD
    neighbour tests do (docs/04 s1). `effective_n` reads that withdrawal as
    "unknown" and applies no discount, which is the optimistic direction -- but
    an honest unknown beats a number measured across a hole.
    """
    adjacent = quarters.diff() == 1
    if int(adjacent.sum()) < MIN_ADJACENT:
        return float("nan")
    return values[adjacent].corr(values.shift(1)[adjacent])


def effective_n(n: int, r1x: float, r1y: float) -> float:
    """Quenouille/Bartlett adjustment for two autocorrelated series, capped at `n`.

    The cap is not cosmetic. The expression is symmetric in the sign of the
    lag-1 product, so a negative product inflates where a positive one
    discounts, without bound and with a pole at -1. An effective sample larger
    than the number of quarters is the correction running backwards:
    persistence can only cost independent observations, never buy them, so `n`
    is the ceiling and a product at or below zero buys nothing back.
    """
    if pd.isna(r1x) or pd.isna(r1y):
        return float(n)
    product = r1x * r1y
    if product <= 0:
        return float(n)
    return max(1.0, float(n) * (1 - product) / (1 + product))


def discounted(r1x: float, r1y: float) -> bool:
    """Whether `effective_n` discounted at all, or handed back `n` unchanged.

    `n_eff == n` has two causes and they read identically in the table: the lag-1
    product came out at or below zero so there was nothing to discount, or `lag1`
    withdrew for want of adjacent pairs and the correction never ran. Either way
    the column holds a raw quarter count under an adjusted column's name.

    Taken from the branch rather than by testing `n_eff == n`, which would call a
    discount too small to survive rounding no discount at all.
    """
    if pd.isna(r1x) or pd.isna(r1y):
        return False
    return bool(r1x * r1y > 0)


# One group must be one series through time, or `sort_values` interleaves two of
# them and every statistic below is computed across a mixture without raising.
# That is a precondition, not a hope, so it is checked rather than assumed.
duplicated = int(gated.duplicated([*SERIES_KEY, "year", "quarter"]).sum())
if duplicated:
    raise ValueError(
        f"{duplicated} rows share a series and a quarter: the screen would correlate "
        "two interleaved series as one. The comparison key has widened -- widen "
        "SERIES_KEY with it."
    )

rows = []
steps = bridged = 0
fewest_adjacent = None
# `dropna=False`: `depth_m` is null for every met parameter and pandas drops null
# group keys by default, which would take `air_temperature` and `wind_speed` --
# two thirds of the grid -- out of the screen without a word.
for key, block in gated.groupby(SERIES_KEY, dropna=False, sort=True):
    series = dict(zip(SERIES_KEY, key, strict=True))
    env_series = series_of(series[column] for column in ENV_SERIES_KEY)
    ordered = block.sort_values(["year", "quarter"])
    for column in ENV_FEATURES:
        feature = column.removesuffix("_anom")
        if (*env_series, feature) in skip:
            continue
        pair = (
            ordered[["year", "quarter", KELP, column]]
            .dropna(subset=[KELP, column])
            .reset_index(drop=True)
        )
        if len(pair) < MIN_PAIRS:
            continue
        quarters = quarter_index(pair["year"], pair["quarter"])
        adjacent = int((quarters.diff() == 1).sum())
        steps += len(pair) - 1
        bridged += (len(pair) - 1) - adjacent
        fewest_adjacent = adjacent if fewest_adjacent is None else min(fewest_adjacent, adjacent)
        # Hoisted out of the `effective_n` call so the same two autocorrelations
        # decide the discount and report whether there was one. Recomputing them
        # for the marker would let the column and the number it describes drift.
        r1_kelp = lag1(pair[KELP], quarters)
        r1_env = lag1(pair[column], quarters)
        rows.append(
            {
                **series,
                "feature": feature,
                "n": len(pair),
                "n_eff": round(effective_n(len(pair), r1_kelp, r1_env), 1),
                "discounted": discounted(r1_kelp, r1_env),
                "pearson_r": round(pair[KELP].corr(pair[column]), 3),
                "spearman_rho": round(pair[KELP].corr(pair[column], method="spearman"), 3),
            }
        )

screen = pd.DataFrame(rows).merge(
    resolution[[*ENV_SERIES_KEY, "feature", "distinct", "verdict"]],
    on=[*ENV_SERIES_KEY, "feature"],
    how="left",
)
screen["low_resolution"] = screen["verdict"].eq("low_resolution")
screen = screen.drop(columns=["verdict"])

# The docs/04 s5 role travels with the cell rather than being applied at the point
# of ranking, so a control is visible as one everywhere it appears instead of only
# where it is excluded. An unmapped parameter raises: `features.json` is what
# decides eligibility, and a parameter it has never heard of has no role to
# default to -- silently treating it as a predictor would enrol it in the pool.
screen["role"] = screen["parameter"].map(ROLES)
unroled = sorted(set(screen.loc[screen["role"].isna(), "parameter"]))
if unroled:
    raise ValueError(
        f"{unroled} are screened but absent from features.json, so nothing says whether "
        "they may be pre-registered. Declare them, or the pool is decided by omission."
    )

print(
    f"{len(screen):,} cells over {screen['polygon_id'].nunique()} polygons, "
    f"{screen['feature'].nunique()} features, {screen['lag'].nunique()} lags"
)
print(f"median n {screen['n'].median():.0f}  ->  median n_eff {screen['n_eff'].median():.0f}")
print()
print(
    f"{bridged:,} of {steps:,} steps between consecutive screened rows ({100 * bridged / steps:.0f}%)"
)
print("cross a quarter gap, and are excluded from the lag-1 autocorrelation rather than")
print(f"counted as adjacent. The thinnest cell still keeps {fewest_adjacent} adjacent pairs, so no")
print(f"cell falls back to the no-discount branch for want of {MIN_ADJACENT} of them.")
print()
undiscounted = int((~screen["discounted"]).sum())
print(f"{undiscounted:,} of {len(screen):,} cells are not discounted at all -- the lag-1 product")
print("came out at or below zero, so `n_eff` is the quarter count unchanged rather than a")
print("measured one. The `discounted` column carries that, because nothing in the number")
print("itself distinguishes the two.")
print()
by_role = screen["role"].value_counts()
print(
    f"{by_role.get('predictor', 0):,} cells are on predictors and {by_role.get('control', 0):,} on"
)
print("controls. Every one of them is screened and reported; only the ranking that feeds")
print("pre-registration is restricted, in s6.")

1,750 cells over 6 polygons, 11 features, 5 lags
median n 93  ->  median n_eff 64

12,092 of 177,838 steps between consecutive screened rows (7%)
cross a quarter gap, and are excluded from the lag-1 autocorrelation rather than
counted as adjacent. The thinnest cell still keeps 20 adjacent pairs, so no
cell falls back to the no-discount branch for want of 3 of them.

287 of 1,750 cells are not discounted at all -- the lag-1 product
came out at or below zero, so `n_eff` is the quarter count unchanged rather than a
measured one. The `discounted` column carries that, because nothing in the number
itself distinguishes the two.

1,420 cells are on predictors and 330 on
controls. Every one of them is screened and reported; only the ranking that feeds
pre-registration is restricted, in s6.


## 4. The matrix

docs/04 §4.1 asks for a lag–feature correlation matrix. Here it is for the pair
the project is actually about: **the La Jolla bed against the water temperature
at LJAC1**, the station inside it.

What to look for, per docs/04 §4.1, is whether the *known physics* shows up —
heat stress at short lags, a cold-water/nitrate association at longer ones. What
to remember is that this is one cell of a grid of several hundred, and that
|r| ≈ 0.2 on ~70 quarters is a hint, not a finding.

In [5]:
def matrix(polygon: str, parameter: str, statistic: str = "pearson_r", **series) -> pd.DataFrame:
    """The lag x feature matrix for one polygon-series pair.

    Refuses rather than averages if the selection covers more than one series.
    `pivot_table` aggregates with the mean, so a second station or a second depth
    on this parameter would silently return the average of two matrices -- which
    reads exactly like one matrix.

    `series` narrows that selection to one: pass whichever `ENV_SERIES_KEY` columns
    it takes -- `site_id`, `depth_m` -- to name the series meant. A null is matched
    with `isna` rather than `==`, since `depth_m` is null for every met parameter
    and `nan == nan` is False.
    """
    cell = screen[(screen["polygon_id"] == polygon) & (screen["parameter"] == parameter)]
    for column, value in series.items():
        cell = cell[cell[column].isna() if pd.isna(value) else cell[column] == value]
    named = cell[ENV_SERIES_KEY].drop_duplicates()
    if len(named) > 1:
        raise ValueError(
            f"{polygon} x {parameter} covers {len(named)} environmental series rather "
            f"than one, so a matrix over it would be their mean: "
            f"{named.to_dict('records')}. Name the series you mean."
        )
    if named.empty:
        raise ValueError(f"{polygon} x {parameter} narrowed by {series} selects no screened cell.")
    return cell.pivot_table(index="feature", columns="lag", values=statistic)


# One labelled matrix per series measuring the parameter, rather than one series
# named as though it stood for the rest. What `matrix` refuses is *averaging* two
# series into a figure that reads like one; printing them separately is the move
# s5 already makes when it drops the key columns that vary, and it is the only way
# two references can be read against each other -- which is the question adding a
# second one asked. A third series appears here by itself rather than silently.
SERIES_SELECTORS = [column for column in ENV_SERIES_KEY if column != "parameter"]

measuring = (
    screen[
        (screen["polygon_id"] == "KELP:LA-JOLLA") & (screen["parameter"] == "sea_water_temperature")
    ][ENV_SERIES_KEY]
    .drop_duplicates()
    .sort_values(ENV_SERIES_KEY)
    .to_dict("records")
)

print("La Jolla kelp area anomaly vs sea water temperature -- Pearson r")
print(f"(comparison sha256:{DIGEST[:16]})")
for series in measuring:
    depth = "no declared depth" if pd.isna(series["depth_m"]) else f"{series['depth_m']:g} m"
    print()
    print(f"{series['site_id']} at {depth}  ({series['env_source']})")
    print(
        matrix(
            "KELP:LA-JOLLA",
            "sea_water_temperature",
            **{column: series[column] for column in SERIES_SELECTORS},
        ).to_string()
    )

La Jolla kelp area anomaly vs sea water temperature -- Pearson r
(comparison sha256:4cde6f9d95207dc1)

SST:LA-JOLLA at no declared depth  (mur_sst)
lag                           0      1      2      3      4
feature                                                    
days_above_20c           -0.379 -0.268 -0.034  0.026  0.012
days_above_23c           -0.104 -0.294 -0.138 -0.029  0.037
days_below_14c            0.056 -0.015 -0.029  0.261  0.291
degree_days_above_18c    -0.335 -0.353 -0.099  0.021  0.036
max                      -0.266 -0.203  0.027  0.052 -0.056
max_spell_above_20c_days -0.293 -0.275  0.016  0.033  0.118
mean                     -0.307 -0.249 -0.036 -0.062 -0.121
min                      -0.194 -0.180  0.050 -0.141 -0.191
p05                      -0.215 -0.163  0.027 -0.068 -0.175
p95                      -0.302 -0.269 -0.048 -0.023 -0.080
variance                 -0.182 -0.182 -0.091 -0.021  0.005

NDBC:46254 at 0.46 m  (ndbc)
lag                           0      1    

## 5. Candidates, ranked

Sorted by |r|, with the two columns that decide whether a coefficient deserves a
second look: `n_eff`, and whether Pearson and Spearman agree.

**Nothing here is a result.** These are the rows to argue about when choosing
what to pre-register in `notebooks/README.md` and carry into docs/04 §4.3.

**`role` says which rows are eligible for that.** docs/04 §5 makes
`air_temperature` and `wind_speed` **controls**: screened and reported, never
pre-registered. Air temperature is largely re-measuring the water — the two
quarterly mean anomalies correlate at r = 0.857 at LJAC1 — and scalar wind speed
averages upwelling-favorable alongshore stress against downwelling-favorable
Santa Ana wind, so its coefficient has no sign to predict. The decision is read
from `features.json` above, not made here, and it is a prior one: it does not
depend on how any cell below came out.

They stay in the table because that is what a control is *for*. If a control
ranks alongside the predictors, the screen is recovering shared seasonality
rather than mechanism — and a screen carrying one predictor family cannot
establish that about itself. The comparison below is the point of keeping them,
so it is printed rather than left to be noticed.

In [6]:
# Drop the key columns that take one value rather than a hardcoded `site_id`, so
# a second station or depth appears in the table by itself instead of being
# folded invisibly into a row that names neither.
constant = {
    column: screen[column].iloc[0]
    for column in SERIES_KEY
    if screen[column].nunique(dropna=False) == 1
}
ranked = (
    screen.assign(
        abs_r=screen["pearson_r"].abs(),
        rank_gap=(screen["pearson_r"] - screen["spearman_rho"]).abs().round(3),
    )
    .sort_values("abs_r", ascending=False)
    .drop(columns=["abs_r", *constant])
)

# `lag` is part of the series key and so arrives before `feature`; read the other
# way round the table separates a parameter from the feature taken off it.
lead = [column for column in ranked.columns if column in SERIES_KEY and column != "lag"]
ranked = ranked[
    [*lead, "feature", "lag", *[c for c in ranked.columns if c not in {*lead, "feature", "lag"}]]
]

print(
    "Every row below is", ", ".join(f"{k}={v}" for k, v in constant.items()) or "(nothing constant)"
)
print()
print("Strongest 15 associations in the screen, controls included and marked:")
print(ranked.head(15).to_string(index=False))
print()
print("Strongest 10 resting on an effective sample of at least 30:")
print(ranked[ranked["n_eff"] >= 30].head(10).to_string(index=False))
print()

# What a control is for. Compared on the same footing the candidate rule uses --
# resolution-flagged cells dropped, `n_eff` floored, Pearson and Spearman agreeing
# -- because a control that ranks well only on cells the pool would have thrown
# out is not evidence of shared seasonality, just evidence of the same artefacts.
comparable = ranked[
    ~ranked["low_resolution"] & (ranked["n_eff"] >= 30) & (ranked["rank_gap"] <= 0.05)
]
print("Do the controls rank with the predictors? On the cells the candidate rule keeps:")
strength = (
    comparable.assign(abs_r=comparable["pearson_r"].abs())
    .groupby(["role", "parameter"])
    .agg(cells=("abs_r", "size"), max_abs_r=("abs_r", "max"), median_abs_r=("abs_r", "median"))
    .round(3)
)
print(strength.to_string())
print()
best = comparable.assign(abs_r=comparable["pearson_r"].abs()).groupby("role")["abs_r"].max()
if {"predictor", "control"} <= set(best.index):
    verdict = (
        "as strongly as"
        if abs(best["control"] - best["predictor"]) < 0.05
        else ("more strongly than" if best["control"] > best["predictor"] else "less strongly than")
    )
    print(f"The strongest control cell is |r| = {best['control']:.2f} and the strongest predictor")
    print(f"cell is |r| = {best['predictor']:.2f}: the controls associate with kelp {verdict}")
    print("the predictors do. Read that as a caution about the screen, not as a finding")
    print("about air temperature or wind -- docs/04 s5 gives the mechanistic reasons those")
    print("two cannot be read as predictors, and no coefficient here revises them.")
else:
    print("One of the two roles has no cell clearing the candidate conditions, so the")
    print("comparison this section exists for cannot be made on this table.")

Every row below is kelp_source=kelpwatch

Strongest 15 associations in the screen, controls included and marked:
         polygon_id         env_source          site_id             parameter  depth_m        feature  lag   n  n_eff  discounted  pearson_r  spearman_rho  distinct  low_resolution      role  rank_gap
     KELP:ENCINITAS sio_shore_stations SIO:LAJOLLA-PIER sea_water_temperature     0.50 days_below_14c    4 161  128.5        True      0.483         0.573        38           False predictor     0.090
     KELP:ENCINITAS sio_shore_stations SIO:LAJOLLA-PIER sea_water_temperature     5.00 days_below_14c    4 161  119.8        True      0.467         0.530        42           False predictor     0.063
  KELP:SOLANA-BEACH sio_shore_stations SIO:LAJOLLA-PIER sea_water_temperature     0.50 days_below_14c    4 161  126.4        True      0.441         0.495        38           False predictor     0.054
     KELP:ENCINITAS               ndbc       NDBC:LJAC1 sea_water_temperature     3

## 6. How much the lag-1 ceiling is worth

§3 says `n_eff` corrects for lag-1 persistence only. This section puts a number
on what that leaves out — **for the candidate cells, and only those.**

The kelp anomaly stays autocorrelated well past one quarter, so a correction
summed over more lags discounts the lag-4 candidates further than the lag-1 one
does. Summing that product over *k* = 1…*K* with the (1 − *k*/*n*) taper:

$$n_{\text{eff}}^{\text{Bartlett}} = \frac{n}{1 + 2\sum_{k=1}^{K}\left(1 - \frac{k}{n}\right) r_k^{(x)} r_k^{(y)}}$$

**A note, not a second statistic, and deliberately not a column.** Across the
whole grid this expression is unbounded above `n` — the taper does not stop
it — and its denominator can reach zero, which leaves it undefined rather than
merely large. docs/04 §1 defers anything data-hungry at this N, and estimating
seventeen autocorrelations from seventy-one quarters and multiplying them in
pairs is data-hungry. The grid-wide measurement, and the argument it settles,
are recorded in
[#35](https://github.com/cweber12/kelp-compare/issues/35). What survives it is a
robustness check on the handful of cells `notebooks/README.md` asks the operator
to weigh — computed here, so a rebuild that moves the table moves the check with
it instead of leaving a stale figure behind.

**Read the sweep, not the headline.** The truncation *K* is a rule of thumb
(⌊*n*/4⌋) and the figure moves with it. A cell that swings several quarters
across plausible *K* is reporting that the tail of the sum is noise, not that it
has a precise effective sample size. Neither number is the truth; the pair of
them brackets it.

**The candidates are selected by the rule `notebooks/README.md` states**, not
named here, so the two cannot disagree without the notebook saying so. The rule
runs over the **predictors only**: docs/04 §5 makes `air_temperature` and
`wind_speed` controls, and a control is never registered whatever its
coefficient. That is a restriction of the pool, applied before the ranking, and
it is stated here so a reader can see the cut rather than infer it from an
absence. §5 reports the control cells it removes.

**The cut counts signals, not cells.** docs/04 §5 defines a signal as one
(feature, lag, polygon), so two eligible cells that agree on all three are one
claim about one bed measured by two instruments and take one place in the list
between them — the strongest of them standing for the rest. Without that, the
list registered the pier and `NDBC:LJAC1` readings of a single bed as two
separate relationships, which is instrumentation counted as replication: the two
pier depths correlate at r = 0.970 and the two stations at 0.785–0.835.

Beds are *not* collapsed. Two beds carrying the same feature at the same lag
stay two signals, because merging them would erase the between-bed comparison
docs/04 §4.5 is built to make — while remaining, as §5 says, two adjacent beds
against shared stations rather than two independent findings. The cell below
prints what each selected signal merged, so the collapse can be checked rather
than taken on trust.

The pool restriction no longer halves the eligible grid, incidentally: it
withheld 330 of 660 when it was written and withholds 330 of 1,750 now, because
the pool grew with the reference series while the two met parameters stayed on
one station.

In [7]:
CANDIDATE_MIN_N_EFF = 30
CANDIDATE_MAX_RANK_GAP = 0.05
CANDIDATE_SIGNALS = 3
K_LADDER = (4, 8, 12, 16)

# docs/04 s5: one signal is one (feature, lag, polygon). Two eligible cells
# agreeing on all three are the same claim about the same bed, measured by
# different instruments -- so they compete for one place in the list rather than
# taking two of three. Cutting the ranking at three *cells* registered the pier
# and LJAC1 readings of one bed as two relationships, which is instrumentation
# counted as replication. Beds stay separate: collapsing them would erase the
# between-bed comparison docs/04 s4.5 exists to make.
SIGNAL_KEY = ["polygon_id", "feature", "lag"]


def autocorrelation(values: pd.Series, quarters: pd.Series, k: int) -> float:
    """Lag-*k* autocorrelation over quarters exactly *k* apart, gaps unbridged.

    `lag1` is this at k = 1, and the check below holds it to that rather than
    trusting it. Generalising the wrong thing matters more here than it did
    there: a positional shift calls every step lag 1 whatever the calendar says,
    and that mistake at seventeen lags compounds seventeen times into the single
    figure this section exists to keep honest.

    Reindexing on the running quarter number puts the holes back as nulls, so
    pairing t against t - k drops a pair whose other end is missing rather than
    reaching across it.
    """
    calendar = pd.Series(values.to_numpy(), index=quarters.to_numpy())
    calendar = calendar.reindex(range(int(quarters.min()), int(quarters.max()) + 1))
    pairs = pd.concat([calendar, calendar.shift(k)], axis=1).dropna()
    if len(pairs) < MIN_ADJACENT:
        return float("nan")
    return pairs.iloc[:, 0].corr(pairs.iloc[:, 1])


def bartlett_n_eff(pair: pd.DataFrame, quarters: pd.Series, column: str, k_max: int) -> float:
    """The higher-order correction, truncated at `k_max` with the (1 - k/n) taper.

    Returns nan rather than a number wherever the expression stops meaning one:
    where a lag keeps too few pairs to estimate, and where the denominator
    reaches zero or below -- which is not a large effective sample but an
    undefined one. Those two are why this is a note and not a column.
    """
    n = len(pair)
    total = 0.0
    for k in range(1, k_max + 1):
        r_kelp = autocorrelation(pair[KELP], quarters, k)
        r_env = autocorrelation(pair[column], quarters, k)
        if pd.isna(r_kelp) or pd.isna(r_env):
            return float("nan")
        total += (1 - k / n) * r_kelp * r_env
    denominator = 1 + 2 * total
    return n / denominator if denominator > 0 else float("nan")


def pair_for(row: pd.Series) -> tuple[pd.DataFrame, pd.Series, str]:
    """The quarters one screened cell was computed over, rebuilt from `gated`.

    Selected on every key column, `depth_m` included, so a second station or a
    second depth on the same parameter cannot be folded in unnoticed -- the trap
    `matrix` refuses by name. The row count is checked back against the screen's
    own, because a selection that quietly matched more or fewer quarters would
    still produce a plausible-looking correction.
    """
    selected = pd.Series(True, index=gated.index)
    for key in SERIES_KEY:
        value = row[key]
        selected &= gated[key].isna() if pd.isna(value) else gated[key] == value
    column = f"{row['feature']}_anom"
    pair = (
        gated[selected]
        .sort_values(["year", "quarter"])[["year", "quarter", KELP, column]]
        .dropna(subset=[KELP, column])
        .reset_index(drop=True)
    )
    if len(pair) != row["n"]:
        raise ValueError(
            f"rebuilt {len(pair)} quarters for a cell the screen measured on {row['n']} -- "
            "the selection does not describe the same series the screen did"
        )
    return pair, quarter_index(pair["year"], pair["quarter"]), column


scored = screen.assign(rank_gap=(screen["pearson_r"] - screen["spearman_rho"]).abs().round(3))
# The role restricts the pool before any coefficient is read, which is what makes
# it a prior decision rather than a filter chosen after seeing the ranking.
pool = scored[scored["role"] == "predictor"]
eligible = pool[
    ~pool["low_resolution"]
    & (pool["n_eff"] >= CANDIDATE_MIN_N_EFF)
    & (pool["rank_gap"] <= CANDIDATE_MAX_RANK_GAP)
]
# Rank the cells, then let the strongest cell in each signal stand for it. The
# cut is applied to *that* frame, so it counts signals; `groupby.head(1)` keeps
# the ranked order, so the strongest signals are still the ones taken.
by_strength = eligible.loc[eligible["pearson_r"].abs().sort_values(ascending=False).index]
signals = by_strength.groupby(SIGNAL_KEY, dropna=False, sort=False).head(1)
candidates = signals.head(CANDIDATE_SIGNALS)

withheld = len(scored) - len(pool)
print(f"{len(pool):,} of {len(scored):,} cells are on predictors and eligible for the pool;")
print(f"{withheld:,} are on controls and are screened but never registered (docs/04 s5).")
print(f"{len(eligible):,} of the eligible cells clear the candidate conditions, and they")
print(f"collapse to {len(signals):,} signals -- one per (feature, lag, polygon). The rule")
print(f"takes the {CANDIDATE_SIGNALS} strongest by |r|; that last step is a cut, not a criterion.")
print()

# What the collapse actually did, per selected signal -- printed rather than
# asserted, because "they register once" is only honest if the cells it merged
# are visible and can be checked for agreeing as closely as s5 claims they do.
print("Each selected signal, and the eligible cells it stands for:")
for _, candidate in candidates.iterrows():
    members = by_strength
    for key in SIGNAL_KEY:
        value = candidate[key]
        members = members[members[key].isna() if pd.isna(value) else members[key] == value]
    lag = candidate["lag"]
    print(
        f"  {candidate['polygon_id']}  {candidate['feature']}  lag {lag}  -- {len(members)} cell(s)"
    )
    for _, member in members.iterrows():
        depth = "" if pd.isna(member["depth_m"]) else f" at {member['depth_m']:g} m"
        stands = "  <-- stands for the signal" if member.name == candidate.name else ""
        print(
            f"      r = {member['pearson_r']:+.3f}  n_eff = {member['n_eff']:5.1f}  "
            f"{member['site_id']}{depth}{stands}"
        )
print()

print("Kelp anomaly autocorrelation per bed -- the persistence `n_eff` does not see:")
persistence = []
for polygon, block in gated.groupby("polygon_id"):
    kelp = (
        block.drop_duplicates(["year", "quarter"])
        .sort_values(["year", "quarter"])[["year", "quarter", KELP]]
        .dropna()
        .reset_index(drop=True)
    )
    kelp_quarters = quarter_index(kelp["year"], kelp["quarter"])
    persistence.append(
        {
            "polygon_id": polygon,
            "quarters": len(kelp),
            "acf_lag1": round(autocorrelation(kelp[KELP], kelp_quarters, 1), 3),
            "acf_lag4": round(autocorrelation(kelp[KELP], kelp_quarters, 4), 3),
        }
    )
print(pd.DataFrame(persistence).to_string(index=False))
print()

headline, sweep = [], []
for _, candidate in candidates.iterrows():
    pair, quarters, column = pair_for(candidate)
    for values in (pair[KELP], pair[column]):
        measured, screened = autocorrelation(values, quarters, 1), lag1(values, quarters)
        if abs(measured - screened) > 1e-12:
            raise ValueError(
                f"lag-1 autocorrelation {measured} here against {screened} in the screen -- "
                "this section would be discounting a cell the screen did not measure"
            )
    rule_of_thumb = len(pair) // 4
    headline.append(
        {
            "polygon_id": candidate["polygon_id"],
            # The series columns travel with the candidate for the reason s5 keeps
            # only the key columns that vary: two references now measure this
            # parameter, so a row naming neither reads as one cell when it is one
            # of two, and the three strongest can be the same signal off two
            # instruments without saying so. That bears on what may be registered.
            "site_id": candidate["site_id"],
            "depth_m": candidate["depth_m"],
            "parameter": candidate["parameter"],
            "feature": candidate["feature"],
            "lag": candidate["lag"],
            "n": len(pair),
            "n_eff": candidate["n_eff"],
            "discounted": candidate["discounted"],
            "K": rule_of_thumb,
            "n_eff_bartlett": round(bartlett_n_eff(pair, quarters, column, rule_of_thumb), 1),
        }
    )
    sweep.append(
        {
            "polygon_id": candidate["polygon_id"],
            "site_id": candidate["site_id"],
            "depth_m": candidate["depth_m"],
            "feature": candidate["feature"],
            "lag": candidate["lag"],
            **{f"K={k}": round(bartlett_n_eff(pair, quarters, column, k), 1) for k in K_LADDER},
        }
    )

headline = pd.DataFrame(headline)
print(f"Candidate cells at the rule-of-thumb truncation K = floor(n/4)   (sha256:{DIGEST[:16]})")
print(headline.to_string(index=False))
print()
print("The same cells across other truncations -- the spread is the point:")
print(pd.DataFrame(sweep).to_string(index=False))
print()

unusable = headline[~(headline["n_eff_bartlett"] <= headline["n"])]
if len(unusable):
    print("The higher-order correction returned no usable figure for:")
    print(unusable.to_string(index=False))
    print("-- above `n`, or undefined. Read `n_eff` alone for these; see issue #35.")
else:
    print("Every candidate's Bartlett figure came back below its own `n`, so all three read")
    print("as written. That is not guaranteed on a rebuilt table: the expression has no")
    print("ceiling and its denominator can cross zero, which is why this stays a note on a")
    print("handful of cells rather than a column across the grid.")

1,420 of 1,750 cells are on predictors and eligible for the pool;
330 are on controls and are screened but never registered (docs/04 s5).
629 of the eligible cells clear the candidate conditions, and they
collapse to 253 signals -- one per (feature, lag, polygon). The rule
takes the 3 strongest by |r|; that last step is a cut, not a criterion.

Each selected signal, and the eligible cells it stands for:
  KELP:ENCINITAS  days_below_14c  lag 4  -- 1 cell(s)
      r = +0.424  n_eff =  50.6  NDBC:LJAC1 at 3.4 m  <-- stands for the signal
  KELP:SOLANA-BEACH  days_below_14c  lag 4  -- 2 cell(s)
      r = +0.406  n_eff = 117.3  SIO:LAJOLLA-PIER at 5 m  <-- stands for the signal
      r = +0.347  n_eff =  49.8  NDBC:LJAC1 at 3.4 m
  KELP:LA-JOLLA  degree_days_above_18c  lag 1  -- 3 cell(s)
      r = -0.353  n_eff =  61.3  SST:LA-JOLLA  <-- stands for the signal
      r = -0.242  n_eff = 124.5  SIO:LAJOLLA-PIER at 0.5 m
      r = -0.220  n_eff = 117.0  SIO:LAJOLLA-PIER at 5 m

Kelp anomaly au

Candidate cells at the rule-of-thumb truncation K = floor(n/4)   (sha256:4cde6f9d95207dc1)
       polygon_id          site_id  depth_m             parameter               feature  lag   n  n_eff  discounted  K  n_eff_bartlett
   KELP:ENCINITAS       NDBC:LJAC1      3.4 sea_water_temperature        days_below_14c    4  71   50.6        True 17            41.7
KELP:SOLANA-BEACH SIO:LAJOLLA-PIER      5.0 sea_water_temperature        days_below_14c    4 161  117.3        True 40            99.5
    KELP:LA-JOLLA     SST:LA-JOLLA      NaN sea_water_temperature degree_days_above_18c    1  95   61.3        True 23            54.2

The same cells across other truncations -- the spread is the point:
       polygon_id          site_id  depth_m               feature  lag   K=4   K=8  K=12  K=16
   KELP:ENCINITAS       NDBC:LJAC1      3.4        days_below_14c    4  49.5  45.2  42.8  41.9
KELP:SOLANA-BEACH SIO:LAJOLLA-PIER      5.0        days_below_14c    4 121.8 120.7 119.9 119.6
    KELP:LA-JOL

## 7. What ranking on |r| costs

§6 takes the three strongest signals **by |r|**, over a pool whose `n_eff` runs
from 30.2 to 161.0. A correlation coefficient is an effect size: it says how
large an association is and nothing about how much record it rests on. So at
equal evidence the rule prefers the shorter series, and at equal |r| it reads a
161-quarter coefficient and a 50-quarter one as the same claim.

That is a defensible rule rather than an oversight — docs/04 §5 asks for effect
sizes rather than p-value collections, and ranking on an effect size is the
consistent way to honour that. But it is a *choice*, it changes the list, and
until now it was unstated. This section measures what it costs rather than
leaving that to be found later.

**Two standardised scales, neither of them a drop-in replacement.**

$$|z| = \left|\operatorname{arctanh} r\right|\sqrt{n_{\text{eff}} - 3}
\qquad\qquad
\mathrm{LCB} = \tanh\left(\max\left(\left|\operatorname{arctanh} r\right| - \frac{1.96}{\sqrt{n_{\text{eff}} - 3}},\; 0\right)\right)$$

Fisher *z* is monotone in the p-value, so ranking on it is the significance
ranking docs/04 §5 forbids, under another name. The lower confidence bound is
not: it stays in correlation units, reports a magnitude rather than a tail
probability, and discounts each coefficient by exactly its own imprecision. If
the rule is ever restated on a standardised scale that is the candidate — and it
is printed here as a diagnostic, not applied.

**Nothing below re-ranks the registration**, and the restraint is the point. The
rule and the list each version of it returns are both on screen now, so choosing
between them from this output would be choosing a rule by the answer it gives —
the selection-on-outcome that pre-registration exists to prevent. The list stays
as §6 ranked it. docs/04 §5 decides, and prospectively.

In [8]:
# Fisher's variance-stabilising transform has variance 1/(n - 3); the offset is
# the transform's, not a tuning choice, and it is why a cell at n_eff just over
# the §6 floor of 30 is scaled by sqrt(27) rather than sqrt(30).
FISHER_OFFSET = 3
CONFIDENCE = 1.96
SCALES = ("abs_r", "abs_z", "lcb")


def fisher_z(r: pd.Series, n_eff: pd.Series) -> pd.Series:
    """|arctanh r| scaled by the record behind it: large where the evidence is large.

    Scaled by `n_eff` rather than `n`, for the reason §3 gives for `n_eff` being
    the column to read. Handing this scale `n` would make the comparison below
    partly about which sample size each scale was given, when it is meant to be
    about the scales.
    """
    return np.arctanh(r).abs() * np.sqrt(n_eff - FISHER_OFFSET)


def lower_bound(r: pd.Series, n_eff: pd.Series) -> pd.Series:
    """The 95% lower confidence limit on |rho|, floored at zero.

    Floored rather than left to go negative: a bound that has crossed zero says
    what one sitting on zero says -- the sign is not established -- so ranking
    those cells against each other below zero would be ranking them on noise.
    """
    return np.tanh(
        (np.arctanh(r).abs() - CONFIDENCE / np.sqrt(n_eff - FISHER_OFFSET)).clip(lower=0)
    )


def scaled(frame: pd.DataFrame) -> pd.DataFrame:
    """The three scales side by side, each ranked over the same rows."""
    out = frame.assign(
        abs_r=frame["pearson_r"].abs(),
        abs_z=fisher_z(frame["pearson_r"], frame["n_eff"]),
        lcb=lower_bound(frame["pearson_r"], frame["n_eff"]),
    )
    for column in SCALES:
        out[f"rank_{column}"] = out[column].rank(ascending=False).astype(int)
    return out


def shown(frame: pd.DataFrame) -> pd.DataFrame:
    """Display columns, rounded -- the ranks are computed on the unrounded values."""
    columns = ["polygon_id", "site_id", "depth_m", "feature", "lag", "pearson_r", "n_eff"]
    return frame[columns + list(SCALES) + [f"rank_{scale}" for scale in SCALES]].round(3)


ranked_signals = scaled(signals)
low, high = ranked_signals["n_eff"].min(), ranked_signals["n_eff"].max()
print(f"The rule ranks {len(ranked_signals):,} signals on |r| alone, over a pool whose")
print(f"`n_eff` runs from {low:.1f} to {high:.1f} -- a {high / low:.0f}x spread.")
print()

# How big the tilt is across the whole pool, rather than only where the cut lands.
# Printed because "the ranking is confounded with record length" is a claim with a
# size, and the size is small: the scales agree in aggregate and separate at the
# top. That is the opposite of a pool-wide slope, and it is how §5 states it.
print("Rank association with record length, over every signal (Spearman):")
for column in SCALES:
    rho = ranked_signals[column].corr(ranked_signals["n_eff"], method="spearman")
    print(f"  {column:>6} against n_eff: {rho:+.3f}")
print("-- |r| leans mildly toward the short records, the standardised scales lean")
print("back. Neither is neutral, and the lean stays mild until the cut.")
print()

for column in SCALES:
    print(f"Top {CANDIDATE_SIGNALS + 2} by {column}, and where each sits on the other scales:")
    print(
        shown(ranked_signals.nsmallest(CANDIDATE_SIGNALS + 2, f"rank_{column}")).to_string(
            index=False
        )
    )
    print()

# What each scale would have registered -- named rather than left to be read off
# the tables, and deliberately not applied. Which scale selects is a docs/04 §5
# question, and it cannot be settled here: the rule and the list it returns are
# both on screen now, so choosing between them from this output would be choosing
# a rule by the answer it gives.
registered = set(ranked_signals.nsmallest(CANDIDATE_SIGNALS, "rank_abs_r").index)
for column in SCALES[1:]:
    alternative = set(ranked_signals.nsmallest(CANDIDATE_SIGNALS, f"rank_{column}").index)
    if alternative == registered:
        print(f"Ranking on {column} returns the registered {CANDIDATE_SIGNALS} unchanged.")
        continue
    print(f"Ranking on {column} would change the list:")
    for index, direction in (
        (registered - alternative, "drops out"),
        (alternative - registered, "enters"),
    ):
        for _, row in ranked_signals.loc[sorted(index)].iterrows():
            print(
                f"  {direction:<9}  {row['polygon_id']:<18} {row['feature']:<22} "
                f"lag {row['lag']}  r = {row['pearson_r']:+.3f}  {column} = {row[column]:.3f}"
            )
print()

# The control check restated on each scale. `notebooks/README.md` quotes the |r|
# line as its standing caution, and the two pools are not comparable on it: every
# control cell is on the one station carrying both met parameters, so the control
# pool stops at an `n_eff` the predictor pool passes by a factor of three.
control_pool = scored[scored["role"] == "control"]
control_eligible = control_pool[
    ~control_pool["low_resolution"]
    & (control_pool["n_eff"] >= CANDIDATE_MIN_N_EFF)
    & (control_pool["rank_gap"] <= CANDIDATE_MAX_RANK_GAP)
]
control_strength = control_eligible.loc[
    control_eligible["pearson_r"].abs().sort_values(ascending=False).index
]
control_signals = scaled(control_strength.groupby(SIGNAL_KEY, dropna=False, sort=False).head(1))

print(f"Predictors against controls, on each scale   (sha256:{DIGEST[:16]})")
print(
    pd.DataFrame(
        [
            {
                "pool": name,
                "signals": len(frame),
                "max n_eff": round(frame["n_eff"].max(), 1),
                **{f"max {column}": round(frame[column].max(), 3) for column in SCALES},
            }
            for name, frame in (("predictor", ranked_signals), ("control", control_signals))
        ]
    ).to_string(index=False)
)
print()
for column in SCALES:
    weakest = ranked_signals.nsmallest(CANDIDATE_SIGNALS, "rank_abs_r")[column].min()
    beating = int((control_signals[column] >= weakest).sum())
    print(
        f"  on {column:>6}: {beating} of {len(control_signals)} control signals reach the "
        f"weakest registered signal ({weakest:.3f})"
    )
print()
print("Those three lines do not say the same thing, and the difference is `n`. The")
print("standardised scales do not simply retire the control caution either: it")
print("clears on abs_z and survives on lcb, where a control still edges the weakest")
print("registered signal. Read the check across the three, not off the |r| line.")

The rule ranks 253 signals on |r| alone, over a pool whose
`n_eff` runs from 30.2 to 161.0 -- a 5x spread.

Rank association with record length, over every signal (Spearman):
   abs_r against n_eff: -0.097
   abs_z against n_eff: +0.247
     lcb against n_eff: +0.164
-- |r| leans mildly toward the short records, the standardised scales lean
back. Neither is neutral, and the lean stays mild until the cut.

Top 5 by abs_r, and where each sits on the other scales:
       polygon_id          site_id  depth_m               feature  lag  pearson_r  n_eff  abs_r  abs_z   lcb  rank_abs_r  rank_abs_z  rank_lcb
   KELP:ENCINITAS       NDBC:LJAC1      3.4        days_below_14c    4      0.424   50.6  0.424  3.122 0.167           1           3         2
KELP:SOLANA-BEACH SIO:LAJOLLA-PIER      5.0        days_below_14c    4      0.406  117.3  0.406  4.606 0.243           2           1         1
    KELP:LA-JOLLA     SST:LA-JOLLA      NaN degree_days_above_18c    1     -0.353   61.3  0.353  2.816 0.

## 8. What this does not say

Carried from docs/04 §6 and the `analysis-review` skill, because a screen is
where over-reading starts:

- **Landsat canopy is a surface expression.** Urchin grazing, subsurface
  condition and predator dynamics are invisible to every dataset in this system,
  so unexplained variance here is ecological rather than statistical.
- **Cloud-gap missingness leans winter** — 9.1% of Q4 and 5.8% of Q1 quarters
  have no cloud-free observation against 0.8% of Q3 — so a screen run on
  non-null quarters is weighted away from winter.
- **Temperature and nitrate proxies are anti-correlated by regional
  oceanography.** A coefficient on a warm feature cannot be read as isolating
  thermal stress from nutrient limitation. That separation is not available from
  this data at all.
- **The 2007–2019 baseline contains the 2014–2016 marine heatwave** on both
  sides, so warm anomalies are damped by the event the analysis most wants to
  detect (docs/04 §3).
- **`air_temperature` and `wind_speed` are controls, not weak predictors.**
  docs/04 §5 withholds them from pre-registration on mechanistic grounds — air
  temperature re-measures the water at r = 0.857, and scalar wind speed averages
  upwelling-favorable stress against its own negation — so no coefficient in
  this notebook argues for or against their exclusion. Their missing Q2 anomaly
  ([#30](https://github.com/cweber12/kelp-compare/issues/30)) and their `n` of
  ~50 against sea water temperature's ~73 now bear on how to read a control,
  not on what may be registered.
- **One predictor family carries the screen, though it is no longer the only
  one.** Of the 629 eligible cells, 620 are `sea_water_temperature`; the wave
  family supplies the other nine — five `wave_significant_height`, four
  `wave_peak_period` — and all nine come from `NDBC:46254` alone. Nine cells
  from one station is not enough for the screen to disagree with itself across
  families, so the control comparison in §5 remains the check that matters on
  whether this is mechanism or shared seasonality. What has changed is the
  station monoculture: the pool now spans `NDBC:LJAC1`, both `SIO:LAJOLLA-PIER`
  depths, `NDBC:46254` and the six `SST:*` series, which is why §5's cut counts
  signals rather than cells.
- **The project's key question is not answered here, and the reason has moved.**
  docs/04 §4.5 compares a project sensor against a public station for kelp at
  increasing distance. Two of the three blockers this cell used to name are
  gone: all six polygons carry geometry, and `observations/` does hold
  project-sensor rows — 10,200 of them reach `comparison.parquet`. Every one
  carries a *null* environmental side rather than an unusable one — the
  distinction §1's attrition table is built on. `PROJ:TIDBIT-1` and
  `PROJ:TIDBIT-2` have one quarter each, 2026 Q3 and not yet over, and the kelp
  record ends at 2026 Q2, so no row in the table reaches that quarter at any
  lag and there was nothing to flag. Behind it sits the blocker that outlasts
  it: one quarter cannot clear a climatology needing ten usable years inside
  2007–2019, so no anomaly is computable and none will be from this record.
  §4.5 is therefore not blocked on ingest but on being specified in
  anomaly space at all
  ([#120](https://github.com/cweber12/kelp-compare/issues/120)). The sensor
  positions were surveyed into `sites.json` on 2026-08-27, so the registry was
  never the obstacle it briefly looked like.